In this notebook, we us the Asynchrone Vinted API for multiple books research. The idea is to look for several books at once with a wishlist and to find a unique Vintie seller of a maximum of wished books. The goal is to save shipping cost. The problem with the Asynchrone setup is that with a list of items to find, without a proxie and a simple code, Vinted will see the list of requests as an attack leading to 403 errors, and blocking requests... Need to be followed. 

In [3]:
import asyncio
from collections import defaultdict
from vinted_scraper import AsyncVintedScraper
import random 

In [ ]:
items_to_find = [
    "the power of habits",
    "the subtle art of not giving a f*ck",
    # "Influence et manipulation", 
    # "La femme de ménage",
    # "Everything is f*cked",
    # "You are a badass",
    # "The Let them theory",
    # "Measure what matters", 
    # "Atomic Habits", 
    # "How to talk to anyone", 
    # "The 100$ startup", 
    # "The art of spending money",
    # "No bullsh*t leadership", 
    # "Think and grow rich", 
    # "Good vibes, good life", 
    # "You can negociate anything", 
    # "The most powerful woman in the room is you", 
    # "Miracle morning", 
    # "What I wish I knew when I was 20", 
    # "Measure what matters", 
    # "You are a badass"
]

async def fetch_item(scraper, searched_item, seller_catalog):
    # print(f"Starting search for {searched_item}") 
    # Scrapping pagination 
    for page in range(1, 5):
        # Define search parameters
        # Getting parameters is possible by clicking on F12 on Vinted page and fetch code and parameters 
        # France: 16; Belgique: 5; Espagne: 13; Italie: 11; Pays-Bas: 20; 
        # Portugal: 29; Luxembourg: 24
        params = {
            "search_text": searched_item,
            "page": page,      # page to search in 
            "per_page": 48,
            # "price_to": 50,
            # "order": "newest_first" # price_low_to_high, relevance, price_high_to_low
            "catalog_ids[]": 2312,  # Books 
            "currency": "EUR",      # forcing euro currency
            "country_ids[]": [16, 5, 13, 11, 20, 29, 24, 9, 2]
                }
       
        try:
             # Perform async search - returns List[VintedItem]
            items = await scraper.search(params)
            
            # If empty page, no more result, stop the loop on pages 
            if not items:
                break

            # adding all users into a dict 
            for item in items:
                user = getattr(item, 'user', None)
                if user and hasattr(user, 'login'):
                    username = user.login
                    # add if found a book with matching a seller 
                    seller_catalog[username].append({
                        "searched_book": item,
                        "title": item.title,
                        "price": item.price,
                        "url": item.url, 
                        "description": item.description,
                    })
            # Avoiding 403 error - wait to change pages 
            await asyncio.sleep(10)
            
        except Exception as e:
            print(f"Error on {page} for {searched_item}: {e}")
            break # go on next book if blocking error             

In [5]:
async def main():
    # Asynchronous Scrapper Initialization 
    scraper = await AsyncVintedScraper.create("https://www.vinted.fr")
    
    # default dictionnary to store matches 
    seller_catalog = defaultdict(list)
    
    print("Starting asynchrone research on Vinted ...\n")
    
    # Async tasks for each item 
    tasks = [
        fetch_item(scraper, book, seller_catalog) 
        for book in items_to_find
    ]
    
    # Executing requests in parallel 
    await asyncio.gather(*tasks)
    delay = random.uniform(5.5, 25.5)
    await asyncio.sleep(delay) # random delay sleep to avoid 403 error
    
    # Closing scrapper to free HTTP connexions in background
    # await scraper.close()
    
    # Display results 
    display_best_matches(seller_catalog)


def display_best_matches(catalog):
    print("\n" + "="*50)
    print("Matching Results")
    print("="*50 + "\n")
    
    # Descending order on Vintie seller's number of available books 
    sorted_sellers = sorted(catalog.items(), key=lambda x: len(x[1]), reverse=True)
    
    has_matches = False
    for username, items in sorted_sellers:
        # Displaying sellers with at least 2 books 
        if len(items) >= 2:
            has_matches = True
            print(f"Vintie : https://www.vinted.fr/membres/{username}")
            print(f"Nombre de correspondances : {len(items)} / {len(items_to_find)}")
            print("Available items:")
            for item in items:
                print(f"  - [{item['searched_book']}] : {item['title']} - {item['price']}€")
                print(f"    Lien : {item['url']}")
            print("-" * 40)
            
    if not has_matches:
        print("No seller found with at least 2 books from the whishlist on the first visited pages.")


In [6]:
# Launching asynch loop 
if __name__ == "__main__":
    await main()

Starting asynchrone research on Vinted ...

Error on 1 for the power of habits: Cannot perform API call to endpoint /api/v2/catalog/items, error code: 403
Error on 1 for the subtle art of not giving a f*ck: Cannot perform API call to endpoint /api/v2/catalog/items, error code: 403

Matching Results

No seller found with at least 2 books from the whishlist on the first visited pages.
